In [90]:
import pandas as pd
import numpy as np

In [91]:
PITCH_LENGTH  = 105
PITCH_WIDTH   = 68
GOAL_X        = 52.5          
GOAL_Y_TOP    =  3.66         
GOAL_Y_BOT    = -3.66        
BALL_ID       = 55            
CONE_WIDTH_M  = 1          
PRESSURE_RADIUS_M = 3   
GK_WIDTH = 1

In [92]:
df = pd.read_csv('../data/full_data.csv')

/var/folders/rr/7hqln5dd7_55vxvrgjhmz1gw0000gn/T/ipykernel_69400/1643287228.py:1: DtypeWarning: Columns (0: game_interruption_after) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/full_data.csv')


In [93]:
# Filter Data
df = df[df['end_type'] == 'shot']
shot_frames = df['frame_end'].unique()
df = df[df['frame'].isin(shot_frames)]

# Combine low occurance possession types into other ones
df.loc[df['team_in_possession_phase_type'] == 'direct', 'team_in_possession_phase_type'] = 'transition'
df.loc[df['team_in_possession_phase_type'] == 'build_up', 'team_in_possession_phase_type'] = 'transition'

In [94]:
df[df['x'] < -52.5]
df[df['x'] > -52.5]

df = df[(df['frame'] != 42202) | (df['match_id'] != 1925299)]
print(df[df['frame'] == 42202]['match_id'])
    

Series([], Name: match_id, dtype: int64)


In [95]:
# Flip the data on the y axis
df2 = df.copy(deep=True)
df2[['y', 'ball_y']] = df2[['y', 'ball_y']] * -1

In [ ]:
def normalise_direction(bx, by, frame_df, period):
    '''
    Changes direction of attack to always be on the right.
    '''
    col = 'direction_player_1st_half' if period == 1 else 'direction_player_2nd_half'
    shooting_team_id  = frame_df['team_id_event'].iloc[0]
    shooter_direction = frame_df[frame_df['team_id'] == shooting_team_id][col].iloc[0]

    if shooter_direction == 'right_to_left':
        bx = -bx
        by = -by
        frame_df = frame_df.copy()
        frame_df['x'] = -frame_df['x']
        frame_df['y'] = -frame_df['y']

    return bx, by, frame_df


def shot_distance(bx, by):
    '''
    Returns the distance of the shot from the center of the goal
    '''
    return np.sqrt((bx - GOAL_X)**2 + by**2)


def shot_angle(bx, by):
    '''
    Returns angle from ball to the two goal posts.
    '''
    a1 = np.arctan2(GOAL_Y_TOP - by, GOAL_X - bx)
    a2 = np.arctan2(GOAL_Y_BOT - by, GOAL_X - bx)
    return abs(a1 - a2)


def defenders_in_cone(bx, by, defenders):
    '''
    Counts the number of defenders between the shooter and the goal within the radius of CONE_WIDTH_M
    '''
    if defenders.empty:
        return 0
    
    # Vector from ball to goal
    dx = GOAL_X - bx
    dy = - by
    length = np.sqrt(dx**2 + dy**2)
    if length < .0001:
        return 0
    
    # Width and length of cone
    ux = dx / length
    uy = dy / length
    perp_x = -uy
    perp_y = ux

    # Defender's position relative to ball
    rel_x  = defenders['x'].values - bx
    rel_y  = defenders['y'].values - by

    # Count the number of defenders within the cone
    in_x   = rel_x * ux + rel_y * uy
    in_y = np.abs(rel_x * perp_x + rel_y * perp_y)
    return int(np.sum((in_x > 0) & (in_x < length) & (in_y < CONE_WIDTH_M)))


def gk_distance_from_line(defending_gk):
    '''
    Measures how far off the goal line the keeper is
    '''
    if defending_gk.empty:
        return np.nan
    return abs(GOAL_X - defending_gk['x'].iloc[0])


def gk_angle_blocked(bx, by, gk_x, gk_y):
    '''
    Measures the percent of shooters angle the keeper is blocking
    '''
    total_angle = shot_angle(bx, by)
    if total_angle < .0001:
        return 0
    
    gk_dist = np.sqrt((bx - gk_x)**2 + (by - gk_y)**2)
    if gk_dist < .0001:
        return 1
    
    gk_angle = 2 * np.arctan((GK_WIDTH / 2) / gk_dist)
    
    fraction_blocked = min(gk_angle / total_angle, 1.0)
    
    return fraction_blocked


def is_pressured(bx, by, defenders):
    '''
    Returns boolean value if there are defenders within a circle w/ PRESSURE_RADIUS_M
    '''
    if defenders.empty:
        return 0
    dists = np.sqrt((defenders['x'].values - bx)**2 +
                    (defenders['y'].values - by)**2)
    return int(np.sum(dists < PRESSURE_RADIUS_M))



# Feature extraction
def extract_shot_features(event_df):
    frame_df = event_df[event_df['frame'] == event_df['frame_end'].iloc[0]].copy()

    bx = frame_df['ball_x'].iloc[0]
    by = frame_df['ball_y'].iloc[0]

    attacking_team_id = frame_df['team_id_event'].iloc[0]
    period = int(frame_df['period'].iloc[0])
    team_ids = frame_df['team_id'].unique()
    defending_ids = [_ for _ in team_ids if _ != attacking_team_id]

    if not defending_ids:
        return None
    defending_team_id = defending_ids[0]

    bx, by, frame_df  = normalise_direction(bx, by, frame_df, period)

    defending_rows = frame_df[frame_df['team_id'] == defending_team_id]
    defending_gk = defending_rows[defending_rows['is_gk'] == True]
    defenders_out = defending_rows[defending_rows['is_gk'] == False]

    if not defending_gk.empty:
        gk_cone = gk_angle_blocked(bx, by, defending_gk['x'].iloc[0], defending_gk['y'].iloc[0])
    else:
        gk_cone = np.nan

    return {
        'event_id': frame_df['event_id'].iloc[0],
        'match_id': frame_df['match_id'].iloc[0],
        'frame': event_df['frame_end'].iloc[0],
        'ball_x': round(bx, 3),
        'ball_y': round(by, 3),
        'distance': round(shot_distance(bx, by), 3),
        'angle': round(shot_angle(bx, by), 4),
        'pressure': is_pressured(bx, by, defenders_out),
        'gk_dist': round(gk_distance_from_line(defending_gk), 3),
        'gk_angle_blocked': gk_cone,
        'defenders_cone': defenders_in_cone(bx, by, defenders_out),
        'one_touch': int(frame_df['one_touch'].iloc[0]),
        'is_header': int(frame_df['is_header'].iloc[0]),
        'phase_type': str(frame_df['team_in_possession_phase_type'].iloc[0]),
        'goal': int(frame_df['lead_to_goal'].iloc[0]),
    }


def build_shot_dataset(df, output_path='shots_clean.csv'):
    shot_df = df[df['end_type'] == 'shot'].copy()
    rows = []
    for (match_id, event_id), group in shot_df.groupby(['match_id', 'event_id']):
        features = extract_shot_features(group)
        if features is not None:
            rows.append(features)

    shots = pd.DataFrame(rows)

    shots = shots.dropna(subset=['gk_dist'])

    shots.to_csv(output_path, index=False)

    return shots

In [97]:
build_shot_dataset(df, output_path='../data/shots_clean.csv')
build_shot_dataset(df2, output_path='../data/shots_clean2.csv')

,event_id,match_id,frame,ball_x,ball_y,distance,angle,pressure,gk_dist,gk_angle_blocked,defenders_cone,one_touch,is_header,phase_type,goal
0,8_103,1886347,4236,28.82,1.06,23.704,0.3061,1,3.42,0.161126,1,1,0,finish,0
1,8_131,1886347,5505,45.76,-13.89,15.439,0.2159,1,2.79,0.461405,1,0,0,finish,0
2,8_244,1886347,10841,31.09,-12.85,24.970,0.2514,0,2.37,0.178647,1,1,0,finish,0
3,8_261,1886347,12202,31.33,-16.13,26.615,0.2194,0,2.11,0.189430,1,0,0,finish,0
4,8_276,1886347,12845,27.21,8.65,26.728,0.2582,0,2.79,0.162442,1,0,0,finish,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,8_663,2017461,58416,38.86,16.51,21.416,0.2206,1,1.68,0.252735,0,0,0,transition,0
221,8_683,2017461,59368,28.10,4.72,24.852,0.2874,1,2.83,0.158125,2,0,0,quick_break,0
222,8_826,2017461,69501,37.08,6.75,16.833,0.3960,1,5.71,0.230444,0,0,0,transition,0
223,8_838,2017461,69858,36.32,-12.12,20.216,0.2911,1,1.88,0.194481,2,1,0,finish,0
